In [8]:
import cv2
import os
# take input if enetred train add the following path to the image and label folder else other path
if input("Enter 'train' for training data or 'validation' for validation data: ").strip().lower() == 'train':
    image_folder = "D:/Meter reading/internship_computer_visions_engineering/data_Set/train/images"
    label_folder = "D:/Meter reading/internship_computer_visions_engineering/data_Set/train/labels"
elif input("Enter 'validation' for validation data: ").strip().lower() == 'validation':
    image_folder = "D:/Meter reading/internship_computer_visions_engineering/data_Set/validation/images"
    label_folder = "D:/Meter reading/internship_computer_visions_engineering/data_Set/validation/labels"
elif input("Enter 'augmented' for augmented training data: ").strip().lower() == 'a':
    image_folder = "../data_Set/train/augmented_images/images"
    label_folder = "../data_Set/train/augmented_images/labels"
else:
    print("Invalid input. Please enter 'train', 'validation', or 'augmented'.")
    exit()


# Ensure the label folder exists
os.makedirs(label_folder, exist_ok=True)

rects = []
drawing = False
start_pt = (0, 0)
current_img = None
clone = None

def draw_rectangle(event, x, y, flags, param):
    global rects, drawing, start_pt, current_img, clone
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        start_pt = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        current_img = clone.copy()
        cv2.rectangle(current_img, start_pt, (x, y), (0, 255, 0), 2)
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        end_pt = (x, y)
        cv2.rectangle(current_img, start_pt, end_pt, (0, 255, 0), 2)
        rects.append((start_pt, end_pt))
        clone = current_img.copy()

cv2.namedWindow("Annotator")
cv2.setMouseCallback("Annotator", draw_rectangle)

# Get list of images
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for filename in image_files:
    img_path = os.path.join(image_folder, filename)
    label_path = os.path.join(label_folder, os.path.splitext(filename)[0] + ".txt")
    
    current_img = cv2.imread(img_path)
    if current_img is None: continue
    clone = current_img.copy()
    rects = []

    print(f"Labeling {filename}... Press 's' to save/next, 'c' to clear, 'q' to quit.")
    
    while True:
        cv2.imshow("Annotator", current_img)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('s'):
            h, w, _ = current_img.shape
            with open(label_path, "w") as f:
                for (pt1, pt2) in rects:
                    x1, y1 = pt1; x2, y2 = pt2
                    x_center = ((x1 + x2) / 2) / w
                    y_center = ((y1 + y2) / 2) / h
                    box_w = abs(x1 - x2) / w
                    box_h = abs(y1 - y2) / h
                    # Assuming class ID 0. Change as needed.
                    f.write(f"0 {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}\n")
            print(f"Saved: {label_path}")
            break # Move to next image
        elif key == ord('c'):
            rects = []
            current_img = cv2.imread(img_path)
            clone = current_img.copy()
        elif key == ord('q'):
            cv2.destroyAllWindows()
            exit()

cv2.destroyAllWindows()
print("All images processed.")

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1284: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvNamedWindow'
